# Voice Cloning Training - Simple Approach
This notebook helps you prepare and test voice cloning with your Speech2Text app recordings.

## ⚠️ Important Note:
Voice cloning technology is rapidly evolving. This notebook shows you:
- ✅ How to **organize and verify** your training data
- ✅ How to **test audio quality**
- ✅ How to **prepare data** for various TTS models
- ✅ **Recommended services** for actual voice cloning

## Recommended Voice Cloning Services:

### 1. **ElevenLabs Voice Cloning** (Best Quality) ⭐⭐⭐⭐⭐
- Website: https://elevenlabs.io
- ✅ Professional quality
- ✅ Only 1 minute of audio needed
- ✅ Multilingual (German, English, Polish, etc.)
- ✅ API available for integration
- 💰 Paid service (free trial available)

### 2. **PlayHT Voice Cloning** ⭐⭐⭐⭐
- Website: https://play.ht
- ✅ Good quality
- ✅ 30 seconds of audio needed
- ✅ API integration
- 💰 Paid service

### 3. **Resemble AI** ⭐⭐⭐⭐
- Website: https://www.resemble.ai
- ✅ Real-time voice cloning
- ✅ Custom models
- 💰 Paid service

### 4. **Local/Free Options:**

**RVC (Retrieval-based Voice Conversion)** ⭐⭐⭐
- Free and open-source
- Requires significant setup
- Best for voice conversion (not TTS)
- Guide: https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI

**OpenVoice** ⭐⭐⭐
- Free and open-source
- Good quality
- Easier setup than RVC
- Guide: https://github.com/myshell-ai/OpenVoice

## What This Notebook Does:

1. ✅ Verify your training data
2. ✅ Analyze audio quality
3. ✅ Prepare data in correct format
4. ✅ Export optimized audio for upload to services
5. ✅ Generate test samples

## Prerequisites:
- Exported training data from Speech2Text app to Google Drive
- At least 10-20 recordings (or 1 minute of clear audio)
- GPU runtime enabled in Colab (optional, for audio processing)

## Steps:
1. Setup and mount Google Drive
2. Load and verify your training data
3. Analyze audio quality
4. Prepare data for voice cloning services
5. Export optimized audio samples

## Step 1: Setup GPU and Mount Google Drive

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected! Please enable GPU: Runtime → Change runtime type → GPU")

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Set the path to your training data (exported from Speech2Text app)
TRAINING_DATA_PATH = '/content/drive/MyDrive/TTS_Voice_Samples'

# Verify the data exists
if os.path.exists(TRAINING_DATA_PATH):
    wav_files = [f for f in os.listdir(TRAINING_DATA_PATH) if f.endswith('.wav')]
    csv_files = [f for f in os.listdir(TRAINING_DATA_PATH) if f.endswith('.csv')]
    
    print(f"\n✅ Found {len(wav_files)} WAV files and {len(csv_files)} CSV metadata file(s)")
    
    # Show CSV file details
    if csv_files:
        csv_file = os.path.join(TRAINING_DATA_PATH, csv_files[0])
        with open(csv_file, 'r', encoding='utf-8') as f:
            line_count = len(f.readlines())
        print(f"   CSV file: {csv_files[0]}")
        print(f"   Contains {line_count} training pairs")
        
        # Read language from CSV
        with open(csv_file, 'r', encoding='utf-8') as f:
            first_line = f.readline().strip()
            if '|' in first_line:
                parts = first_line.split('|')
                if len(parts) >= 3:
                    detected_lang = parts[2].strip()
                    print(f"   Detected language: {detected_lang}")
    else:
        print("⚠️ No CSV metadata file found!")
        print("   Make sure to export training data from the app using the 💾 button.")
else:
    print("\n❌ Training data folder not found. Please export data from the app first.")
    print(f"   Expected location: {TRAINING_DATA_PATH}")

## Step 2: Install Audio Processing Tools

In [ ]:
# Install audio processing libraries
print("📥 Installing audio processing tools...\n")

!pip install librosa soundfile pydub numpy matplotlib

print("\n✅ Audio tools installed successfully!")

In [ ]:
# Import libraries
import librosa
import soundfile as sf
import numpy as np
import matplotlib.pyplot as plt
from pydub import AudioSegment
import os

print("✅ All libraries imported successfully!")
print("\n📊 Ready to analyze your audio recordings")

## Step 3: Analyze Your Voice Recordings

In [ ]:
import os
import librosa
import numpy as np
import matplotlib.pyplot as plt

# Read metadata from CSV
csv_files = [f for f in os.listdir(TRAINING_DATA_PATH) if f.endswith('.csv')]

if not csv_files:
    print("❌ No CSV file found! Please export data from the app.")
else:
    csv_file = os.path.join(TRAINING_DATA_PATH, csv_files[0])
    
    # Parse CSV
    recordings = []
    with open(csv_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('|')
            if len(parts) >= 3:
                filename, text, lang = parts[0], parts[1], parts[2]
                recordings.append({
                    'file': filename,
                    'text': text,
                    'lang': lang
                })
    
    print(f"✅ Found {len(recordings)} recordings")
    print(f"   Language: {recordings[0]['lang'] if recordings else 'Unknown'}")
    print(f"\n📝 Sample recordings:")
    for i, rec in enumerate(recordings[:5]):
        audio_path = os.path.join(TRAINING_DATA_PATH, rec['file'])
        if os.path.exists(audio_path):
            y, sr = librosa.load(audio_path, sr=None)
            duration = len(y) / sr
            print(f"   {i+1}. {rec['file']} ({duration:.1f}s)")
            print(f"      Text: {rec['text'][:60]}...")
    
    # Calculate total duration
    total_duration = 0
    for rec in recordings:
        audio_path = os.path.join(TRAINING_DATA_PATH, rec['file'])
        if os.path.exists(audio_path):
            y, sr = librosa.load(audio_path, sr=None)
            total_duration += len(y) / sr
    
    print(f"\n⏱️ Total audio duration: {total_duration:.1f} seconds ({total_duration/60:.1f} minutes)")
    
    if total_duration < 60:
        print("   ⚠️ Less than 1 minute. Recommended: 1-5 minutes for best quality")
    elif total_duration < 180:
        print("   ✅ Good amount of data for voice cloning")
    else:
        print("   ✅ Excellent amount of data!")

## Step 4: Audio Quality Analysis

In [ ]:
# Analyze first recording in detail
if recordings:
    sample_audio = os.path.join(TRAINING_DATA_PATH, recordings[0]['file'])
    y, sr = librosa.load(sample_audio, sr=None)
    
    print("🔍 Detailed Audio Analysis:")
    print(f"\n📊 Technical Details:")
    print(f"   Sample Rate: {sr} Hz")
    print(f"   Duration: {len(y)/sr:.2f} seconds")
    print(f"   Bit Depth: 16-bit PCM (from WAV)")
    print(f"   Channels: Mono")
    
    # Check quality
    print(f"\n✅ Quality Checks:")
    
    if sr >= 22050:
        print(f"   ✅ Sample rate: {sr} Hz (Excellent)")
    elif sr >= 16000:
        print(f"   ⚠️ Sample rate: {sr} Hz (Acceptable, but 22050+ is better)")
    else:
        print(f"   ❌ Sample rate: {sr} Hz (Too low, re-record at 22050+ Hz)")
    
    # Check for silence
    rms = librosa.feature.rms(y=y)[0]
    silence_threshold = 0.01
    silence_ratio = np.sum(rms < silence_threshold) / len(rms)
    
    if silence_ratio < 0.3:
        print(f"   ✅ Silence: {silence_ratio*100:.1f}% (Good)")
    else:
        print(f"   ⚠️ Silence: {silence_ratio*100:.1f}% (Too much silence, trim audio)")
    
    # Check volume
    max_amplitude = np.max(np.abs(y))
    if max_amplitude > 0.8:
        print(f"   ✅ Volume level: Good (max: {max_amplitude:.2f})")
    elif max_amplitude > 0.3:
        print(f"   ⚠️ Volume level: Low (max: {max_amplitude:.2f}) - consider amplifying")
    else:
        print(f"   ❌ Volume level: Too low (max: {max_amplitude:.2f}) - re-record louder")
    
    # Plot waveform
    plt.figure(figsize=(14, 4))
    plt.subplot(1, 2, 1)
    plt.plot(np.linspace(0, len(y)/sr, len(y)), y)
    plt.title('Waveform')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.grid(True)
    
    # Plot spectrogram
    plt.subplot(1, 2, 2)
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Spectrogram')
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Tip: Check the visualizations above for audio quality")

## Step 5: Prepare Audio for Voice Cloning Services

In [ ]:
from IPython.display import Audio, display
import shutil

# Create export directory
EXPORT_DIR = "/content/voice_clone_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

print("📦 Preparing audio files for voice cloning services...\n")

# Option 1: Concatenate all recordings into one file (for ElevenLabs, etc.)
print("🎵 Creating single combined audio file...")

combined_audio = []
silence = np.zeros(int(0.5 * 22050))  # 0.5 second silence between clips

for rec in recordings:
    audio_path = os.path.join(TRAINING_DATA_PATH, rec['file'])
    if os.path.exists(audio_path):
        y, sr = librosa.load(audio_path, sr=22050)  # Resample to 22050 Hz
        combined_audio.append(y)
        combined_audio.append(silence)

if combined_audio:
    combined = np.concatenate(combined_audio)
    combined_path = os.path.join(EXPORT_DIR, "voice_sample_combined.wav")
    sf.write(combined_path, combined, 22050)
    
    duration = len(combined) / 22050
    print(f"   ✅ Combined file created: voice_sample_combined.wav")
    print(f"      Duration: {duration:.1f} seconds ({duration/60:.1f} minutes)")
    print(f"      Perfect for: ElevenLabs, PlayHT, Resemble AI")
    
    # Play preview
    print("\n🔊 Preview (first 10 seconds):")
    preview = combined[:int(10*22050)]
    display(Audio(preview, rate=22050))

# Option 2: Export individual high-quality samples
print("\n📂 Exporting individual samples...")
individual_dir = os.path.join(EXPORT_DIR, "individual_samples")
os.makedirs(individual_dir, exist_ok=True)

for i, rec in enumerate(recordings[:10]):  # Export first 10
    audio_path = os.path.join(TRAINING_DATA_PATH, rec['file'])
    if os.path.exists(audio_path):
        # Load and resample
        y, sr = librosa.load(audio_path, sr=22050)
        
        # Normalize volume
        y = y / np.max(np.abs(y)) * 0.9
        
        # Save
        output_path = os.path.join(individual_dir, f"sample_{i+1:02d}.wav")
        sf.write(output_path, y, 22050)

print(f"   ✅ Exported {min(10, len(recordings))} individual samples")
print(f"      Perfect for: Training datasets, comparison")

print(f"\n✅ All files prepared in: {EXPORT_DIR}")

## Step 6: Export to Google Drive

In [ ]:
from datetime import datetime
import shutil

# Export to Google Drive
drive_export_dir = '/content/drive/MyDrive/Voice_Clone_Ready'
os.makedirs(drive_export_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
final_export_dir = os.path.join(drive_export_dir, f"voice_samples_{timestamp}")

print("📤 Exporting to Google Drive...\n")

# Copy all prepared files
shutil.copytree(EXPORT_DIR, final_export_dir)

# Create README with instructions
readme_content = f"""# Voice Clone Audio Samples

Created: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
Language: {recordings[0]['lang'] if recordings else 'Unknown'}
Total recordings: {len(recordings)}
Total duration: {total_duration:.1f} seconds

## Files:

### voice_sample_combined.wav
- Combined audio of all your recordings
- Perfect for quick upload to voice cloning services
- Use with: ElevenLabs, PlayHT, Resemble AI

### individual_samples/
- Individual audio samples
- Each normalized to 22050 Hz
- Use for: Custom training, comparison

## Recommended Services:

### 1. ElevenLabs (Best Quality)
1. Go to: https://elevenlabs.io
2. Sign up / Login
3. Click "Voice Lab" → "Add Voice"
4. Upload: voice_sample_combined.wav
5. Enter name and description
6. Click "Add Voice"
7. Test with sample text

### 2. PlayHT
1. Go to: https://play.ht
2. Sign up / Login
3. Navigate to "Voice Cloning"
4. Upload: voice_sample_combined.wav
5. Follow the wizard

### 3. Local Options (Advanced)
- RVC: https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI
- OpenVoice: https://github.com/myshell-ai/OpenVoice

## Quality Tips:
- Use the combined file for simplicity
- Individual samples for fine-tuning
- More audio = better quality
- Recommend 1-5 minutes total

## Next Steps:
1. Download the combined WAV file
2. Upload to your chosen service
3. Test the cloned voice
4. If quality is not good enough, record more samples in the app
"""

with open(os.path.join(final_export_dir, "README.txt"), 'w', encoding='utf-8') as f:
    f.write(readme_content)

print(f"✅ Export complete!")
print(f"\n📁 Location: {final_export_dir}")
print(f"\n📝 Files exported:")
print(f"   - voice_sample_combined.wav (main file for upload)")
print(f"   - individual_samples/ ({min(10, len(recordings))} files)")
print(f"   - README.txt (instructions)")

print(f"\n🎯 Next Steps:")
print(f"   1. Open Google Drive")
print(f"   2. Navigate to: Voice_Clone_Ready/voice_samples_{timestamp}")
print(f"   3. Download voice_sample_combined.wav")
print(f"   4. Upload to ElevenLabs or PlayHT")
print(f"   5. Enjoy your cloned voice! 🎉")

## Step 5: Test the Fine-Tuned Model

In [ ]:
from IPython.display import Audio, display

# Load the fine-tuned model (if you completed Step 4B)
# Otherwise this will use zero-shot cloning from Step 4A

try:
    # Try to load fine-tuned model
    finetuned_tts = TTS(model_path=f"{OUTPUT_MODEL_DIR}/best_model.pth", config_path=f"{OUTPUT_MODEL_DIR}/config.json").to("cuda" if torch.cuda.is_available() else "cpu")
    print("✅ Loaded fine-tuned model")
    use_finetuned = True
except:
    print("⚠️ Fine-tuned model not found, using zero-shot cloning")
    finetuned_tts = tts
    use_finetuned = False

# Test sentences in different languages
test_sentences = {
    'de': "Hallo! Dies ist meine geklonte Stimme. Die Qualität ist durch das Fine-Tuning deutlich besser geworden.",
    'en': "Hello! This is my cloned voice. The quality has improved significantly through fine-tuning.",
    'pl': "Cześć! To jest mój sklonowany głos. Jakość znacznie się poprawiła dzięki dostrojeniu."
}

test_text = test_sentences.get(LANGUAGE, test_sentences['en'])

print(f"\n🤖 Generating test audio...")
print(f"   Text: {test_text}")

output_path = "/content/finetuned_test.wav"

if use_finetuned:
    # Fine-tuned model doesn't need speaker_wav
    finetuned_tts.tts_to_file(
        text=test_text,
        file_path=output_path
    )
else:
    # Zero-shot needs reference audio
    reference_audio = os.path.join(TRAINING_DATA_PATH, recordings[0]['file'])
    finetuned_tts.tts_to_file(
        text=test_text,
        speaker_wav=reference_audio,
        language=LANGUAGE,
        file_path=output_path
    )

print("\n🔊 Generated audio:")
display(Audio(output_path))

## Step 6: Export Model to Google Drive

In [ ]:
import shutil
from datetime import datetime

# Create export directory
export_dir = '/content/drive/MyDrive/XTTS_Voice_Models'
os.makedirs(export_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_export_dir = f"{export_dir}/my_voice_{LANGUAGE}_{timestamp}"
os.makedirs(model_export_dir, exist_ok=True)

if use_finetuned and os.path.exists(OUTPUT_MODEL_DIR):
    # Export fine-tuned model
    print("📦 Exporting fine-tuned model...")
    
    # Copy all model files
    for file in os.listdir(OUTPUT_MODEL_DIR):
        if file.endswith(('.pth', '.json', '.txt')):
            src = os.path.join(OUTPUT_MODEL_DIR, file)
            dst = os.path.join(model_export_dir, file)
            shutil.copy2(src, dst)
            print(f"   ✅ {file}")
    
    # Copy test audio
    if os.path.exists('/content/finetuned_test.wav'):
        shutil.copy2('/content/finetuned_test.wav', f"{model_export_dir}/test_sample.wav")
        print(f"   ✅ test_sample.wav")
    
    print(f"\n✅ Fine-tuned model exported to: {model_export_dir}")
else:
    # Export reference audio for zero-shot
    print("📦 Exporting reference audio for zero-shot cloning...")
    
    reference_audio = os.path.join(TRAINING_DATA_PATH, recordings[0]['file'])
    shutil.copy2(reference_audio, f"{model_export_dir}/speaker_reference.wav")
    
    # Save metadata
    with open(f"{model_export_dir}/info.txt", 'w') as f:
        f.write(f"Language: {LANGUAGE}\n")
        f.write(f"Model: XTTS-v2 Zero-Shot\n")
        f.write(f"Reference text: {recordings[0]['text']}\n")
    
    print(f"\n✅ Reference audio exported to: {model_export_dir}")
    print("   Use this with XTTS-v2 for zero-shot cloning.")

## Step 7: Create Usage Instructions

In [ ]:
# Create README with usage instructions
readme_content = f"""# Voice Clone Model - {LANGUAGE}

Created: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
Language: {LANGUAGE}
Model: {"Fine-tuned XTTS-v2" if use_finetuned else "XTTS-v2 Zero-Shot"}

## Usage in Python:

```python
from TTS.api import TTS
import torch

# Load model
{'tts = TTS(model_path="best_model.pth", config_path="config.json")' if use_finetuned else 'tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")'}
tts = tts.to("cuda" if torch.cuda.is_available() else "cpu")

# Generate speech
{'tts.tts_to_file(text="Your text here", file_path="output.wav")' if use_finetuned else 'tts.tts_to_file(text="Your text here", speaker_wav="speaker_reference.wav", language="' + LANGUAGE + '", file_path="output.wav")'}
```

## Quality:

{"- Fine-tuned on your voice recordings" if use_finetuned else "- Zero-shot cloning using reference audio"}
- Best for {LANGUAGE} language
- Sample audio: test_sample.wav

## Integration with Android:

For Android integration, you'll need to:
1. Convert the model to ONNX format
2. Use ONNX Runtime on Android
3. Or use a server-based approach with this Python model

## Further Improvement:

To improve quality:
1. Record more diverse sentences (20-50 clips)
2. Ensure consistent audio quality
3. Run Step 4B again with more epochs
"""

with open(f"{model_export_dir}/README.md", 'w', encoding='utf-8') as f:
    f.write(readme_content)

print("✅ Created README.md with usage instructions")
print(f"\n📁 Complete export location: {model_export_dir}")

## Next Steps:

1. **Download the trained model** (`.onnx` and `.onnx.json` files)
2. **Integrate with your Android app** using Piper TTS library
3. **Test the voice** in your app
4. **Continue improving**: Record more data and run Step 6b to refine the model

## Fine-Tuning Tips:

- **Start small**: 10-20 recordings are enough to start
- **Incremental improvement**: Add 5-10 more recordings and re-train
- **Consistent quality**: Same environment, same microphone
- **Clear pronunciation**: Speak naturally and clearly
- **Vary your sentences**: Use different sentence structures
- **Monitor quality**: Test after each training session

## Why Fine-Tuning is Better:

- ✅ **Faster Results**: Get a working model in 30 minutes instead of 10+ hours
- ✅ **Less Data**: 10-20 samples work well, not 100+
- ✅ **Better Quality**: Pre-trained model already knows language patterns
- ✅ **Iterative**: Keep improving by adding more data
- ✅ **Cost Efficient**: Less GPU time needed on Colab

## Troubleshooting:

- **If quality is poor after first training**: Add 10 more recordings and re-train
- **If voice sounds robotic**: Increase recording length (3-8 seconds per clip)
- **If training fails**: Check that all WAV files have matching TXT files
- **To improve specific words**: Record more sentences with those words